## Consolidación Arquitectónica (K=12) y Primer Comité de Inferencia (V3)

Los objetivos de estenotebook son: instanciar la arquitectura óptima de **12 clústeres** sobre el corpus deduplicado (`prospectos_unicos_semanticos_15.csv`), evitando los problemas de sobreajuste de iteraciones anteriores; ejecutar el *Prompt* de Abstracción Semántica sobre un comité de inferencia de 3 modelos avanzados (DeepSeek-R1, Llama3 y Gemma-3); e implementar las primeras rutinas de limpieza de expresiones regulares para estandarizar las salidas JSON de los LLMs.

In [1]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_unicos_semanticos_15.csv")

df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 12 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 165.65it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 869/869 [13:49<00:00,  1.05it/s]


In [19]:
df_parrafos.to_csv("prospectos_clusters12.csv", index=False)

In [4]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_clusters12.csv")

In [20]:
df_parrafos[df_parrafos["cluster"]==0]["parrafo_anonimizado"]

27       Este medicamento contiene menos de 1 mmol de s...
30       En la prevención de la embolia provocada por u...
40       No tome una dosis doble para compensar las dos...
103      Este medicamento contiene menos de 1 mmol de s...
104      La dosis recomendada de este medicamento para ...
                               ...                        
27736    Adolescentes (a partir de 12 años) y niños ≥ 6...
27737    Segunda dosis: 6,5 mg/kg (sin exceder 400 mg),...
27738    Tercera dosis: 6,5 mg/kg (sin exceder 400 mg),...
27739    Cuarta dosis: 6,5 mg/kg (sin exceder 400 mg), ...
27741    Debe emplearse la dosis mínima eficaz y no deb...
Name: parrafo_anonimizado, Length: 1940, dtype: str

## Preparación de prompt

In [ ]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)

  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres propios de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿Puedo conducir después de tomar [MEDICAMENTO]?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me tengo que tomar [MEDICAMENTO]?".
  3. Este fragmento Los demás componentes son celulosa microcristalina (E460), carboximetilalmidón sódico (Tipo A) de patata, povidona K90 (E1201)" responde a "¿Cuál es la composición de este medicamento?"
  4. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Identifica el tema que se repite en el 80% de los fragmentos y descarta los detalles que solo aparecen en uno solo y genera la PREGUNTA DIRECTA, GENERAL Y NO ESPECÍFICA que un paciente normal (no experto) haría. 

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [6]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=1000) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

In [7]:
import requests

def listar_modelos_disponibles():
    url_base = "https://wiig.dia.fi.upm.es/ollama/api/tags"
    try:
        response = requests.get(url_base, timeout=10)
        if response.status_code == 200:
            modelos = [m['name'] for m in response.json().get('models', [])]
            print(f" Modelos disponibles: {modelos}")
            return modelos
        else:
            print(f" No se pudo listar modelos. Status: {response.status_code}")
    except Exception as e:
        print(f" Error de conexión: {e}")
    return []

modelos_reales = listar_modelos_disponibles()

 Modelos disponibles: ['llama3_evaluator:latest', 'gpt-oss:120b', 'deepseek-r1:70b', 'gemma3:27b', 'gemma3:12b', 'gemma3:4b', 'gemma3:1b', 'mistral:7b', 'qwen3:30b', 'qwen3:14b', 'llama3.1:70b', 'llama3.1:8b', 'deepseek-r1:32b', 'nemotron-3-nano:30b', 'deepseek-v3.2:cloud', 'qwen2.5:32b', 'qwen3:1.7b', 'qwen3:8b', 'llama3_easy:latest', 'llama3_hard:latest', 'llama3_medium:latest', 'llama3_medium_Open:latest', 'llama3_easy_Open:latest', 'llama3_hard_Open:latest', 'llama3:latest']


In [5]:
anotadores = ["llama3:latest", "llama3_medium:latest", "llama3_hard:latest"]
resultados = []

In [14]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_12_topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando llama3:latest...
[INST] <<SYS>>
{
  "pregunta_del_paciente": "¿Cuál es la dosis recomendada para adultos?"
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Cuál es la dosis adecuada para mí según mi edad y peso?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Cuánto tomaré si tengo sobrepeso?"
}
 Procesando Cluster 1...
  -> Consultando llama3:latest...
{
  "pregunta_del_paciente": "¿Puedo seguir tomando [MEDICAMENTO] si estoy embarazada o intentando quedarme embarazada?"
}
  -> Consultando llama3_medium:latest...
{
  "pregunta_del_paciente": "¿Puedo estar embarazada y tomar [MEDICAMENTO] al mismo tiempo?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Puedo seguir usando [MEDICAMENTO] si estoy embarazada o intentando quedarme embarazada?"
}
 Procesando Cluster 2...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Cómo debo tomar estos comprimidos y qué sucede si

### Análisis de Resultados: Validación Topológica vs. Limitación Cognitiva
La evaluación de las respuestas generadas por el comité Llama3 arroja conclusiones mixtas. Los resultados demuestran que **la topología de K=12 es correcta**, pero **el modelo de inferencia es insuficiente**.

#### El Éxito Arquitectónico (Validación de K=12)
La agrupación en 12 clústeres ha demostrado ser un éxito rotundo a nivel de estructuración de la información. A diferencia de lo ocurrido con K=14, los temas ya no están fragmentados:
* **Consenso Absoluto:** Temas vitales como el Embarazo (Cluster 1), la Dosis (Cluster 0) y los Coágulos Sanguíneos (Cluster 7) presentan una convergencia casi perfecta entre los modelos, formulando preguntas claras, directas y útiles para el paciente.

#### La Limitación Cognitiva (Por qué necesitamos otros modelos)
A pesar de que los datos están bien agrupados, los modelos Llama3 (8B) demuestran falta de capacidad para seguir las restricciones negativas del *System Prompt* ("PROHIBIDO mencionar nombres"):
1. **Alucinaciones Léxicas:** Los modelos se "contaminan" con ejemplos específicos de los párrafos. Se inventan el uso de *levofloxacino* (Cluster 3), *Micralax* y la *hidroxicloroquina* (Clusters 8 y 10). 
2. **Inestabilidad de Formato:** Se detectan fallos en la generación de JSON. En el Cluster 11, `llama3_medium` altera las claves (`"pREGUNTA_DEL_PACIENTE"`) y omite corchetes de cierre.

### Prueba con modelos diferentes

Con la intención de probar el rendimiento de los 12 clusters, probamos con 3 modelos con diferentes capacidades.


In [19]:
import time

anotadores = ["deepseek-r1:32b", "llama3_hard:latest", "gemma3:27b"]
resultados = []

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("preguntas_v3_12topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando deepseek-r1:32b...
ERROR: 500 Server Error: Internal Server Error for url: https://wiig.dia.fi.upm.es/ollama/v1/chat/completions
  -> Consultando llama3_hard:latest...
The repetitive theme in the 80% of the fragments is "dose" or "dosage". Therefore, I'll generate a question that a normal patient (non-expert) might ask, and respond to it in the format you requested.

Here's the answer:

{
"pregunta_del_paciente": "¿Cuál es la dosis correcta para tomar [MEDICAMENTO]?"
}
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Cómo debo tomar este medicamento y cuál es la dosis correcta?"
}
```
 Procesando Cluster 1...
  -> Consultando deepseek-r1:32b...
```json
{
    "pregunta_del_paciente": "¿Puedo tomar este medicamento si estoy embarazada o estoy amamantando?"
}
```
  -> Consultando llama3_hard:latest...
{
  "pregunta_del_paciente": "¿Es seguro tomar [MEDICAMENTO] durante el embarazo o lactancia?"
}
  -> Consultando gemma3:27b...

### Auditoría de Inferencia: Diagnóstico de Comportamiento de los LLMs
Al analizar las salidas del comité V3 sobre el espacio K=12, se diagnostican tres comportamientos clave:

##### 1. Colapso de Infraestructura (DeepSeek-R1 32B)
El modelo DeepSeek-R1 experimentó fallos críticos (*Error 500: Internal Server Error*) en 5 de las 12 peticiones (Clusters 4, 6, 8, 9). Esto demuestra que, aunque el modelo es capaz de razonar adecuadamente cuando funciona, su alta demanda computacional provoca *timeouts* inasumibles en el servidor de inferencia. 

##### 2. Ruptura de Formato y "Conversational Filler" (Llama3_hard 8B)
A pesar de la instrucción estricta de responder *exclusivamente* en formato JSON, Llama3 sufre "fugas de instrucción" (Instruction Bleed). En los Clústeres 0 y 3, el modelo incluye preámbulos conversacionales (*"The repetitive theme in the 80%..."*, *"La tema que se repite..."*). 

##### 3. El Estándar Semántico (Gemma-3 27B)
Gemma-3 demuestra una adherencia perfecta al *Role-Play* ("Paciente no experto") y extrae la intención universal sin alucinar nombres de fármacos. Sin embargo, envuelve consistentemente sus respuestas en bloques de código Markdown.

In [ ]:
import pandas as pd
import ast
import json
import re

def limpiar_y_extraer_json(texto):
    """Extrae el contenido de un JSON incluso si viene envuelto en Markdown."""
    if pd.isna(texto) or str(texto).strip() == "":
        return ""
    
    # Eliminar bloques de código markdown (```json ... ```)
    texto_limpio = re.sub(r'```json|```', '', str(texto)).strip()
    
    try:
        # Intentamos cargar como JSON
        data = json.loads(texto_limpio)
        return data.get("pregunta_del_paciente", texto_limpio)
    except:
        # Si falla el parseo JSON, intentamos literal_eval o devolvemos el texto original
        try:
            data = ast.literal_eval(texto_limpio)
            return data.get("pregunta_del_paciente", texto_limpio)
        except:
            return texto_limpio


df_raw = pd.read_csv("preguntas_v3_12topicos.csv")

df_orig = pd.DataFrame()
df_orig['cluster_id'] = df_raw['cluster']

modelos = {
    'res_deepseek-r1_32b': 'deepseek-r1:32b',
    'res_llama3_hard_latest': 'llama3.1:70b',
    'res_gemma3_27b': 'gemma3:27b'
}

for col_csv, nombre_final in modelos.items():
    if col_csv in df_raw.columns:
        df_orig[nombre_final] = df_raw[col_csv].apply(limpiar_y_extraer_json)

df_orig.set_index('cluster_id', inplace=True)

df_orig.to_csv("preguntas_originales_v3.csv")

print("Archivo 'preguntas_originales_v3.csv' guardado con éxito.")
print(df_orig.head())

Archivo 'preguntas_originales_v3.csv' guardado con éxito.
                                              deepseek-r1:32b  \
cluster_id                                                      
0           ERROR: 500 Server Error: Internal Server Error...   
1           ¿Puedo tomar este medicamento si estoy embaraz...   
2           ¿Qué debo hacer si me olvido de tomar una dosi...   
3           ¿Cuál es la probabilidad de desarrollar una er...   
4           ERROR: 500 Server Error: Internal Server Error...   

                                                 llama3.1:70b  \
cluster_id                                                      
0           The repetitive theme in the 80% of the fragmen...   
1           ¿Es seguro tomar [MEDICAMENTO] durante el emba...   
2           ¿Cuál es el uso correcto de los comprimidos/me...   
3           La tema que se repite en el 80% de los fragmen...   
4           ¿Cuál es el efecto secundario más común del us...   

                              